# Pump It Up — Part 1 of 3: Data Preparation

**Competition:** [DrivenData — Pump It Up: Data Mining the Water Table](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/)

**Pipeline position:** `[ 01 Data Preparation ] → 02 Model Training → 03 Interactive Annexes`

This notebook covers everything that happens **before any model is trained**: data loading, full exploratory data analysis, and the leakage-free feature engineering pipeline. Its outputs — engineered feature matrices and the fitted pipeline artifacts — are written to `artifacts/` and consumed by `02_model_training.ipynb` and `03_interactive_annexes.ipynb`.

---

## Table of Contents

**SETUP**
- Section 1 — Data loading and initial configuration

**EXPLORATORY DATA ANALYSIS (EDA)**
- Section 2.1 — Variable inventory (types, cardinality, missing values)
- Section 2.2 — Target variable distribution (class imbalance)
- Section 2.3 — Numerical variables: distributions and problematic values
- Section 2.4 — Geographic variables
- Section 2.5 — Categorical variables: cardinality and class distribution
- Section 2.6 — High-cardinality variables
- Section 2.7 — Missing value analysis
- Section 2.8 — Numerical variables: anomaly detection
- Section 2.9 — Relationship between categorical variables and target
- Section 2.10 — Automated EDA with Sweetviz

**FEATURE ENGINEERING**
- Section 3.1 — Transformation decisions
- Section 3.2 — Fit imputers and encoders on train only
- Section 3.3 — `feature_pipeline.py` — single source of truth
- Section 3.4 — Transformation verification

**HANDOFF**
- Section 4 — Persist artifacts for notebooks 02 and 03


---

## 1. Data loading and initial configuration

In [ ]:
# Library installation for THIS notebook only.
# Part 1 needs: pandas, numpy, matplotlib, seaborn, scikit-learn (core stack),
# plus sweetviz (automated EDA report) and pyarrow (Parquet artifact handoff).
import subprocess, sys, importlib

def install(pkg, import_name=None):
    mod = import_name or pkg.replace('-', '_')
    try:
        importlib.import_module(mod)
        print(f'  OK  {pkg} (already installed)')
    except ImportError:
        print(f'  Installing {pkg}...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                            '--quiet', '--no-warn-script-location'],
                           capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "ERROR"}  {pkg}')

install('sweetviz')
install('pyarrow')


In [ ]:
# Standard data analysis and visualisation libraries.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42


In [ ]:
# ── IB Brand Style + shared plotting helpers ─────────────────────────────────
# Loads ib_style.py if present (same folder). Falls back to defaults otherwise.
import sys, os
import numpy as np
import matplotlib.pyplot as plt

try:
    sys.path.insert(0, os.getcwd())
    from ib_style import (
        apply_style, apply_seaborn_style, styled_fig,
        MPL, cmap_blue, cmap_orange, importance_colors,
        plot_confusion_matrix as ib_cm, style_geo_ax,
        get_gradio_theme, get_gradio_css, html_ai_assistant,
    )
    apply_style()
    apply_seaborn_style()
    IB_STYLE = True
    print('OK  IB brand style loaded and applied.')
except (ImportError, FileNotFoundError) as e:
    IB_STYLE = False
    print(f'WARN  ib_style.py not found ({e}) — default matplotlib style.')
    def styled_fig(nrows, ncols, figsize=(8, 6), title=None, subtitle=None):
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
        if title:
            fig.suptitle(title, fontsize=14, fontweight='bold')
        return fig, axes
    MPL = {'text':'#111111','bg':'white','bg_ax':'#fafafa','text2':'#555555',
           'functional':'#4C9BE8','needs_repair':'#E87C4C','non_functional':'#E74C3C',
           'blue':'#4C9BE8','orange':'#E87C4C','green':'#4CB87A',
           'grid':'#dddddd','teal':'#16a085','purple':'#8e44ad','red':'#E74C3C'}

C_FUNC   = MPL['functional']
C_REPAIR = MPL['needs_repair']
C_NONFUN = MPL['non_functional']
BG       = MPL['bg']
BG_AX    = MPL['bg_ax']
TEXT2    = MPL['text2']

STATUS_PALETTE = {
    'functional':              C_FUNC,
    'functional needs repair': C_REPAIR,
    'non functional':          C_NONFUN,
}


In [ ]:
# The competition data is distributed across four files:
import pandas as pd
# - train_features.csv: the 40 descriptive variables for each pump in the training set
# - train_labels.csv: the target variable (status_group) for each pump in the training set
# - test_features.csv: the same 40 variables for the pumps we need to predict
# - submission_format.csv: the exact structure required for the submission file
# The join key between files is the 'id' field.

from pathlib import Path

def locate_data_folder():
    current = Path.cwd()
    for folder in [current] + list(current.parents):
        if (folder / 'train_features.csv').exists() and (folder / 'train_labels.csv').exists() and (folder / 'test_features.csv').exists():
            return folder
        fallback = folder / 'Mejora_NeedsRepair'
        if (fallback / 'train_features.csv').exists() and (fallback / 'train_labels.csv').exists() and (fallback / 'test_features.csv').exists():
            return fallback
    raise FileNotFoundError('Data directory with train_features.csv, train_labels.csv and test_features.csv not found')

DATA_DIR = locate_data_folder()

train_features = pd.read_csv(DATA_DIR / 'train_features.csv')
train_labels   = pd.read_csv(DATA_DIR / 'train_labels.csv')
test_features  = pd.read_csv(DATA_DIR / 'test_features.csv')
submission_fmt = pd.read_csv(DATA_DIR / 'submission_format.csv')

# Merge features and labels into a single working DataFrame
df = train_features.merge(train_labels, on='id')

print(f'Training set: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Test set:     {test_features.shape[0]:,} rows x {test_features.shape[1]} columns')
df.head(3)


---

## 2. Data exploration

Before building any model it is important to understand what each variable contains, its type, distribution and quality issues. This phase drives all subsequent transformation decisions.


### 2.1 Variable inventory

In [ ]:
# A summary table is generated with the data type, number of unique values,
import pandas as pd
# null count and missing percentage for each column.
# This allows identifying at a glance which variables have issues
# and what type they are (high-cardinality categorical, numeric with zeros, etc.).

summary = pd.DataFrame({
    'dtype':     df.dtypes.astype(str),
    'n_unique':  df.nunique(),
    'n_nulls':   df.isnull().sum(),
    'pct_nulls': (df.isnull().mean() * 100).round(2),
    'example':   df.iloc[0]
})
summary = summary.sort_values('pct_nulls', ascending=False)
display(summary)


In [ ]:
# Variables can be grouped into four categories by nature:
#
# CONTINUOUS NUMERICAL: amount_tsh, gps_height, longitude, latitude, population
# DISCRETE NUMERICAL: construction_year, num_private, region_code, district_code
# CATEGORICAL (low cardinality, <= 20 values): basin, region, water_quality,
#   quantity, source, waterpoint_type, payment, extraction_type_class, management_group
# CATEGORICAL (high cardinality, > 20 values): funder, installer, wpt_name,
#   subvillage, ward, scheme_name, lga, extraction_type, management
# DATE: date_recorded
# IDENTIFIER: id, recorded_by (single value: no predictive information)

num_cols = ['amount_tsh', 'gps_height', 'longitude', 'latitude',
            'num_private', 'region_code', 'district_code', 'population',
            'construction_year']

cat_low  = ['basin', 'region', 'public_meeting', 'scheme_management',
            'permit', 'extraction_type_group', 'extraction_type_class',
            'management_group', 'payment', 'water_quality', 'quality_group',
            'quantity', 'source_class', 'waterpoint_type_group']

cat_high = ['funder', 'installer', 'wpt_name', 'subvillage', 'ward',
            'lga', 'scheme_name']

print('Numerical variables:', num_cols)
print('\nLow-cardinality categoricals:', cat_low)
print('\nHigh-cardinality categoricals:', cat_high)


### 2.2 Target variable distribution

In [ ]:
# Class imbalance is a critical aspect in multi-class classification.
# If unaddressed, the model may learn to predict majority classes well
# while ignoring minority ones — yielding misleading accuracy but little practical value.

label_counts = df['status_group'].value_counts()
label_pct    = df['status_group'].value_counts(normalize=True) * 100

print('Target variable distribution:')
for label in label_counts.index:
    print(f'  {label:<30} {label_counts[label]:>6,}  ({label_pct[label]:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C9BE8', '#E87C4C', '#4CB87A']
bars = ax.bar(label_counts.index, label_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for bar, pct in zip(bars, label_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Class distribution in the training set', fontsize=12)
ax.set_ylabel('Number of pumps')
ax.set_xlabel('')
_axes_to_style = globals().get('ax', None)
if _axes_to_style is None:
    _axes_to_style = globals().get('ax_bar', None)
if _axes_to_style is None:
    _axes_list = [ax]
else:
    try:
        _axes_list = list(np.array([_axes_to_style]).flatten())
    except Exception:
        _axes_list = [_axes_to_style]
for _ax in _axes_list:
    if _ax is None:
        continue
    _ax.grid(False)
    for _sp in _ax.spines.values(): _sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


**Conclusion — Class distribution:** The dataset shows significant imbalance: `functional` accounts for 54.3%, `non functional` for 38.4%, and `functional needs repair` for only 7.3%. This last group is the most critical practically (an undetected pump needing repair will eventually fail), but also the hardest to predict. Sections 5 and 5.3 address this imbalance with `class_weight` and evaluate the specific recall for this class. The overall accuracy metric alone will be misleading if the minority class is not predicted well.


The `functional needs repair` class represents only 7.3% of the total, compared to 54.3% for `functional`. This imbalance causes models to systematically confuse it with the other two classes, and explains why overall accuracy can be high even when performance on that class is poor.


### 2.3 Numerical variables: distributions and problematic values

In [ ]:
# In field data, zeros often indicate missing values rather than real measurements.
# This is clearly the case for longitude (cannot be 0 in Tanzania),
# latitude, construction_year (pumps with year=0 do not exist) and population.
# The problem is quantified here before deciding how to handle it.

problematic_zeros = {
    'longitude':          (df['longitude'] == 0).sum(),
    'latitude':           (df['latitude']  == 0).sum(),
    'gps_height':         (df['gps_height'] == 0).sum(),
    'construction_year':  (df['construction_year'] == 0).sum(),
    'population':         (df['population'] == 0).sum(),
    'amount_tsh':         (df['amount_tsh'] == 0).sum(),
}

print('Rows with zero values that likely represent missing data:')
for col, n in problematic_zeros.items():
    pct = n / len(df) * 100
    print(f'  {col:<22} {n:>6,}  ({pct:.1f}%)')


In [ ]:
# Distributions of the four main numerical variables are visualised.
#
# gps_height and construction_year: KDE by class. Their distributions are readable
# and show clear class separation, indicating predictive power.
#
# population and amount_tsh: KDE is not used because 60-83% of values are zero,
# which collapses the x-axis into an unreadable vertical line. Instead the
# percentage of zeros per class is shown — the most relevant information:
# the higher the proportion of zeros in a class, the more that missing data
# pattern correlates with pump status.

palette = {'functional': '#4C9BE8',
           'functional needs repair': '#E8C84C',
           'non functional': '#E87C4C'}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# ── gps_height: KDE by class ──────────────────────────────────────────────────
ax = axes[0]
for label, color in palette.items():
    subset = df[df['status_group'] == label]['gps_height']
    subset = subset[subset > 0]
    if len(subset) > 10:
        subset.plot.kde(ax=ax, label=label, color=color, linewidth=1.8)
ax.set_title('gps_height', fontsize=11)
ax.set_xlabel('')
ax.legend(fontsize=8)

# ── construction_year: KDE by class ──────────────────────────────────────────
ax = axes[1]
for label, color in palette.items():
    subset = df[df['status_group'] == label]['construction_year']
    subset = subset[subset > 0]
    if len(subset) > 10:
        subset.plot.kde(ax=ax, label=label, color=color, linewidth=1.8)
ax.set_title('construction_year', fontsize=11)
ax.set_xlabel('')
ax.legend(fontsize=8)

# ── population: % zeros by class ─────────────────────────────────────────────
# Pumps classified as "functional needs repair" have the highest % of population=0 (41%),
# which may indicate less monitored areas or poorer data recording.
ax = axes[2]
zero_pct = {label: (df[df['status_group'] == label]['population'] == 0).mean() * 100
            for label in palette}
bars = ax.bar(list(zero_pct.keys()), list(zero_pct.values()),
              color=list(palette.values()), edgecolor='white', width=0.5)
ax.set_title('population: % zero values by class', fontsize=10)
ax.set_ylabel('% records with population = 0')
ax.tick_params(axis='x', rotation=12)
for i, (label, pct) in enumerate(zero_pct.items()):
    ax.text(i, pct + 0.5, f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

# ── amount_tsh: % zeros by class ─────────────────────────────────────────────
# Most informative of the two: non-functional pumps have 83% zeros in amount_tsh
# vs 61% for functional — a 22 percentage-point gap is a clear predictive signal.
ax = axes[3]
zero_tsh = {label: (df[df['status_group'] == label]['amount_tsh'] == 0).mean() * 100
            for label in palette}
bars = ax.bar(list(zero_tsh.keys()), list(zero_tsh.values()),
              color=list(palette.values()), edgecolor='white', width=0.5)
ax.set_title('amount_tsh: % zero values by class', fontsize=10)
ax.set_ylabel('% records with amount_tsh = 0')
ax.tick_params(axis='x', rotation=12)
for i, (label, pct) in enumerate(zero_tsh.items()):
    ax.text(i, pct + 0.5, f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Numerical variable distributions by pump status', fontsize=12)
for _ax in (np.array([axes]).flatten() if hasattr(axes,'__len__') else [axes]):
    _ax.grid(False)
    for _sp in _ax.spines.values(): _sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_numeric_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


**Conclusion — Numerical variables by class:** `construction_year` shows the greatest class separation: functional pumps concentrate in more recent years (post-2000 peak) while non-functional ones spread toward earlier decades. `gps_height` shows more modest but perceptible separation. `population` and `amount_tsh` have very high zero rates (36-83%) representing masked missing values — these zeros will be converted to NaN during feature engineering to prevent them from distorting distributions and model statistics.


It can be observed that `construction_year` shows visible class separation: older pumps have a higher probability of being non-functional. This variable will have predictive relevance. `gps_height` and `population` show more overlapping distributions, though they still contribute some information.


### 2.4 Geographic variables

In [ ]:
# Left panel: geographic scatter map coloured by status.
# Right panel: percentage of non-functional pumps by region, sorted ascending.
#
# Geographic filter (f — improvement):
#   Tanzania spans roughly lon 29-41 E, lat 1-12 S.
#   Original filter (lon>10, lat<-0.5) included valid border pumps but excluded
#   a small number of records with lon between 10-29 that may be valid.
#   Revised filter uses lon>28 & lat<-0.1 to tighten to Tanzanian territory
#   while preserving Kagera region pumps near lon~29, lat~-1.

geo = df[(df['longitude'] > 28) & (df['latitude'] < -0.1)].copy()
n_filtered = len(df) - len(geo)
if n_filtered > 0:
    print(f'Note: {n_filtered} records excluded by geographic filter '
          f'(likely invalid/placeholder coordinates).')

fig, axes = plt.subplots(1, 2, figsize=(17, 8))

scatter_palette = {
    'functional':              '#4C9BE8',
    'non functional':          '#E87C4C',
    'functional needs repair': '#E8C84C',
}
for label, color in scatter_palette.items():
    subset = geo[geo['status_group'] == label]
    axes[0].scatter(subset['longitude'], subset['latitude'],
                    c=color, s=3, alpha=0.3, label=label)
axes[0].set_title('Geographic Distribution by Status', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=4, fontsize=9)

geo_nf = (geo.groupby('region')['status_group']
            .apply(lambda x: (x == 'non functional').mean() * 100)
            .sort_values(ascending=True))

norm_vals = (geo_nf.values - geo_nf.min()) / (geo_nf.max() - geo_nf.min())
bar_colors = plt.cm.RdYlGn_r(norm_vals)

axes[1].barh(geo_nf.index, geo_nf.values, color=bar_colors, edgecolor='white', height=0.7)
axes[1].set_title('% Non-Functional Pumps by Region', fontsize=12, fontweight='bold')
axes[1].set_xlabel('% non-functional pumps')
for i, v in enumerate(geo_nf.values):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8.5)

for _ax in (np.array([axes]).flatten() if hasattr(axes,'__len__') else [axes]):
    _ax.grid(False)
    for _sp in _ax.spines.values(): _sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_geo.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Records with valid GPS coordinates: {len(geo):,} of {len(df):,}')
top3_bad  = geo_nf.tail(3).index.tolist()
top3_good = geo_nf.head(3).index.tolist()
print(f'Regions with highest failure rate: {", ".join(reversed(top3_bad))}')
print(f'Regions with lowest failure rate: {", ".join(top3_good)}')


**Conclusion — Geographic distribution:** Geographic location has a direct and measurable impact on pump status. Southern regions (Lindi 64.2%, Mtwara 62.4%, Tabora 54.4%) have failure rates three times higher than the north (Iringa 19.5%, Arusha 26.3%). This reflects structural differences in maintenance investment, accessibility and average installation age. As a result, `latitude`, `longitude` and the target-encoded `region_fail_rate` will be the highest-importance variables in the final model.


The map confirms that non-functional pumps are not randomly distributed: there are areas with high failure concentration (south and south-west) and high density of functional pumps (north and north-east). The bar chart quantifies this by region: **Lindi and Mtwara** exceed 60% non-functional pumps while **Iringa and Arusha** remain below 27%.

This has direct implications for the model: geographic variables (latitude, longitude, region) will be highly predictive, and the `region_fail_rate` target encoding captures exactly this pattern. As a next step, this variable will be created during preprocessing.


### 2.5 Categorical variables: cardinality and class distribution

In [ ]:
# Proporción de cada clase target por variable categórica. Pastel, sin grid.

vars_to_plot = ['quantity', 'waterpoint_type_group', 'extraction_type_class',
                'water_quality', 'payment', 'basin']

if IB_STYLE:
    fig, axes = styled_fig(3, 2, figsize=(14, 16),
                            title='Proportion of statuses by category (key categorical variables)')
else:
    fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

C_FUNC_P   = '#7db7f0'
C_REPAIR_P = '#fdb36b'
C_NONFUN_P = '#f48d8d'
ordered_status = ['functional', 'functional needs repair', 'non functional']
pastel_colors  = [C_FUNC_P, C_REPAIR_P, C_NONFUN_P]

for idx, (ax, col) in enumerate(zip(axes, vars_to_plot)):
    df_plot = globals().get("df")
    if df_plot is None:
        if "train_features" in globals() and "train_labels" in globals():
            df_plot = pd.concat([train_features.reset_index(drop=True),
                                train_labels.reset_index(drop=True)], axis=1)
        else:
            raise NameError("df is not defined. Run the data-loading cell before this plot.")

    ct = pd.crosstab(df_plot[col], df_plot['status_group'], normalize='index') * 100
    cols_order = [c for c in ordered_status if c in ct.columns]
    ct_sorted  = ct[cols_order].sort_values('functional', ascending=True)
    bars_bottom = np.zeros(len(ct_sorted))
    for status, color in zip(cols_order, pastel_colors):
        if status in ct_sorted.columns:
            ax.barh(ct_sorted.index, ct_sorted[status],
                     left=bars_bottom, color=color,
                     label=status, edgecolor='none', height=0.65)
            bars_bottom += ct_sorted[status].values
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18),
               ncol=3, fontsize=7.5, framealpha=0)
    ax.set_title(col.replace('_', ' ').title(), pad=32, fontsize=10, fontweight='bold')
    ax.set_xlabel('Percentage (%)', fontsize=9)
    ax.set_xlim(0, 100)
    ax.grid(False)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(labelsize=8)

plt.tight_layout(pad=2.5)
plt.savefig('fig_cat_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()
import pandas as pd


Several relevant findings:

- `quantity`: pumps with water classified as `dry` are almost always non-functional. Those with `enough` are mostly functional — highly discriminative.
- `payment`: pumps with no maintenance charge (`never pay`) have a much higher proportion of non-functional ones.
- `extraction_type_class`: simpler or manual extraction types tend to fail more often.


### 2.6 High-cardinality variables

### 2.7 Missing value analysis

We identify variables with missing data. The treatment strategy depends on percentage and variable type:
- **< 5%** — safe imputation with mode or median
- **5-30%** — impute and consider adding a binary indicator feature
- **> 50%** — consider dropping the variable (unreliable)


In [ ]:
# The percentage of missing values per column is calculated and visualised
import pandas as pd
# with reference lines marking decision thresholds.
# Bar colour indicates severity: green (low), orange (medium), red (high).
# Missing values here are exclusively MCAR/MAR type in categorical variables:
# funder, installer, scheme_management, permit and public_meeting.

missings     = df.drop(columns=['status_group']).isnull().sum()
missings_pct = (missings / len(df)) * 100
missing_df   = (pd.DataFrame({'N_missing': missings, 'Percentage': missings_pct})
                .pipe(lambda d: d[d['N_missing'] > 0])
                .sort_values('Percentage', ascending=True))

print('Variables with missing values:')
print('-' * 48)
for col, row in missing_df.iterrows():
    level = 'HIGH  ' if row['Percentage'] > 30 else 'MEDIUM' if row['Percentage'] > 5 else 'LOW   '
    print(f'  {col:<25}: {row["N_missing"]:>6,} ({row["Percentage"]:>5.1f}%)  {level}')

fig, ax = plt.subplots(figsize=(9, max(3, len(missing_df) * 0.55)))
bar_colors = ['#e74c3c' if p > 30 else '#f39c12' if p > 5 else '#2ecc71'
              for p in missing_df['Percentage']]
missing_df['Percentage'].plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white')
ax.set_xlabel('% missing values', fontsize=11)
ax.set_title('Percentage of Missing Values by Variable', fontsize=13, fontweight='bold')
ax.axvline(x=5,  color='green',  linestyle='--', alpha=0.6, label='5% (low)')
ax.axvline(x=30, color='orange', linestyle='--', alpha=0.6, label='30% (medium-high)')
ax.axvline(x=50, color='red',    linestyle='--', alpha=0.5, label='50% (consider dropping)')
ax.legend(fontsize=9)
for i, (col, row) in enumerate(missing_df.iterrows()):
    ax.text(row['Percentage'] + 0.3, i, f'{row["Percentage"]:.1f}%', va='center', fontsize=9)
for _ax in (np.array([ax]).flatten() if hasattr(ax,'__len__') else [ax]):
    _ax.grid(False)
    for _sp in _ax.spines.values(): _sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_missings.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('Treatment decisions:')
print('  scheme_name (48.5%): drop — too many missing values, low utility')
print('  funder / installer / scheme_management / permit: impute with "Unknown"')
print('  subvillage / public_meeting: impute with mode or "Unknown"')


**Conclusion — Missing values:** Only `scheme_name` exceeds the critical 30% threshold (48.5%), making it unreliable — it will be dropped. The remaining variables with missing values (funder, installer, scheme_management, permit, public_meeting) are between 5% and 7% — manageable by imputing with the "Unknown" category and adding binary flags (`has_funder`, `has_scheme_mgmt`) so the model can distinguish records with and without information, since the absence itself may be predictive of pump status.


The chart shows that **scheme_name** (48.5%) is the only variable with a critical level of missing values — more than half the records have no value, making it unreliable as a predictor. The rest (funder, installer, scheme_management, permit, public_meeting) are between 5% and 7%, manageable by imputing with the "Unknown" category.

As a next step, `scheme_name` will be dropped during feature engineering and binary flags (`has_funder`, `has_scheme_mgmt`) will be added so the model can distinguish records with and without that information, since the absence itself may be predictive.


### 2.8 Numerical Variables — Anomaly Detection

We inspect numerical distributions for impossible or suspicious values that actually represent masked missing data. This is very common in real-world datasets: instead of leaving NaN, a 0 is recorded that looks valid but is not (a pump cannot be at longitude 0, nor built in year 0).


In [ ]:
# Histogramas de las 8 variables numéricas — degradado pastel + sin cuadrícula.
import matplotlib.colors as mcolors
import numpy as np

# Fallback de colores si no han sido definidos en celdas previas
if 'C_NONFUN' not in globals():
    C_NONFUN = '#e74c3c'   # rojo
if 'C_REPAIR' not in globals():
    C_REPAIR = '#f39c12'   # naranja
if 'TEXT2' not in globals():
    TEXT2 = '#888888'      # gris

num_vars = ['amount_tsh', 'gps_height', 'longitude', 'latitude',
            'population', 'construction_year', 'district_code', 'region_code']

if IB_STYLE:
    fig, axes = styled_fig(2, 4, figsize=(20, 9),
                            title='Numerical Variable Distributions — Anomaly Detection',
                            subtitle='Colour intensity = relative frequency  ·  Warm tones = high peaks')
else:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

_pastel_low  = np.array([0.49, 0.72, 0.94])   # azul pastel
_pastel_high = np.array([0.99, 0.70, 0.42])   # naranja pastel

def hist_gradient_colors(counts):
    cmin, cmax = counts.min(), counts.max()
    rng = max(cmax - cmin, 1)
    return [mcolors.to_hex((1-t)*_pastel_low + t*_pastel_high)
            for t in (counts - cmin) / rng]

for i, col in enumerate(num_vars):
    ax   = axes[i]
    data = df[col].dropna()
    counts, edges = np.histogram(data, bins=40)
    colors_h = hist_gradient_colors(counts)
    for left, right, h, color in zip(edges[:-1], edges[1:], counts, colors_h):
        ax.bar(left, h, width=(right-left)*0.92,
               color=color, align='edge', linewidth=0)
    ax.set_title(col, fontsize=10, fontweight='bold',
                 color=MPL['text'] if IB_STYLE else '#111')
    ax.tick_params(labelsize=8)
    ax.set_facecolor(MPL['bg_ax'] if IB_STYLE else '#fafafa')
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    pct_zeros = (df[col] == 0).mean() * 100
    if pct_zeros > 50:
        alert_color = C_NONFUN; note = f'⚠ {pct_zeros:.0f}% zeros'
    elif pct_zeros > 20:
        alert_color = C_REPAIR; note = f'{pct_zeros:.0f}% zeros'
    else:
        alert_color = TEXT2;    note = f'{pct_zeros:.0f}% zeros' if pct_zeros > 0 else 'No zeros'
    ax.text(0.97, 0.97, note, transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color=alert_color,
            fontweight='bold' if pct_zeros > 20 else 'normal')

fig.patch.set_facecolor(MPL['bg'] if IB_STYLE else 'white')
plt.tight_layout(pad=2.5)
plt.savefig('fig_num_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

print('Summary of anomalous values:')
print('-' * 65)
anomalies = {
    'construction_year': (df['construction_year'] == 0).sum(),
    'longitude':         (df['longitude'] < 1).sum(),
    'latitude':          (df['latitude'] > -0.5).sum(),
    'population':        (df['population'] == 0).sum(),
    'amount_tsh':        (df['amount_tsh'] == 0).sum(),
    'gps_height':        (df['gps_height'] == 0).sum(),
}
for col, count in anomalies.items():
    pct = count / len(df) * 100
    print(f'  {col:<22}: {count:>6,} records ({pct:.1f}%) with 0 or impossible value')
print()
print('Zeros in coordinates and construction year are masked missing values.')
print('They will be replaced with NaN before imputing with the median of the combined set.')

**Conclusion — Numerical anomalies:** Zeros in `amount_tsh` (70%), `construction_year` (35%), `gps_height` (34%) and `population` (36%) are masked missing values, not real ones. A pump with `longitude=0` would be in the Indian Ocean; one with `construction_year=0` was never built. These zeros will be treated as NaN before imputing with the median calculated on the combined train+test set, preventing them from distorting statistics and negatively affecting model quality.


The histograms reveal the **masked zeros** pattern in six critical variables. The most extreme case is `amount_tsh`: 70% of records have value zero, which is physically impossible for an operational pump. Similarly, `construction_year=0` appears in 35% of pumps and `longitude/latitude=0` places those pumps in the Indian Ocean.

All these zeros will be converted to `NaN` before imputation with the median, preventing them from distorting distributions and model statistics. The derived variable `pump_age` will be calculated only on records with a valid construction year, and negative values (recording errors) will also be treated as NaN.


### 2.9 Relationship between Categorical Variables and Target

We visualise how pump status distribution varies across the most relevant categorical variables using 100% stacked bars. This format is clearer than horizontal bars for comparing proportions across many categories and allows immediate identification of which values are associated with a higher failure rate.


In [ ]:
# 100% stacked bars por variable categórica — pastel, sin grid.
import pandas as pd

vars_cat_target = [
    ('quantity',              'Water availability'),
    ('waterpoint_type',       'Waterpoint type'),
    ('extraction_type_class', 'Extraction class'),
    ('payment',               'Payment system'),
    ('water_quality',         'Water quality'),
    ('source_class',          'Water source class'),
    ('management_group',      'Management group'),
    ('basin',                 'Hydrographic basin'),
]

if IB_STYLE:
    fig, axes = styled_fig(4, 2, figsize=(18, 26),
                            title='Pump Status Distribution by Categorical Variables')
else:
    fig, axes = plt.subplots(4, 2, figsize=(18, 24))
axes = axes.flatten()

ordered_status = ['functional', 'functional needs repair', 'non functional']
COLORS_STACKED = ['#7db7f0', '#fdb36b', '#f48d8d']

for i, (col, title) in enumerate(vars_cat_target):
    ct = (pd.crosstab(df[col], df['status_group'], normalize='index') * 100)
    oc = [c for c in ordered_status if c in ct.columns]
    ct_sorted = ct[oc].sort_values('functional', ascending=True)
    bottom = np.zeros(len(ct_sorted))
    for status, color in zip(oc, COLORS_STACKED):
        axes[i].barh(ct_sorted.index, ct_sorted[status],
                      left=bottom, color=color, label=status,
                      edgecolor='none', height=0.72)
        bottom += ct_sorted[status].values
    axes[i].set_title(f'Status by {title}', pad=30, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('% of pumps', fontsize=9)
    axes[i].legend(loc='upper center', bbox_to_anchor=(0.5, 1.2),
                    ncol=3, fontsize=7, framealpha=0,
                    labels=['Functional', 'Needs repair', 'Non functional'])
    axes[i].grid(False)
    for sp in axes[i].spines.values(): sp.set_visible(False)
    axes[i].tick_params(labelsize=8)

plt.tight_layout(pad=2.5)
plt.savefig('fig_cat_stacked.png', dpi=150, bbox_inches='tight')
plt.show()


**Conclusion — Categorical variables vs target:** The strongest patterns are: (1) `quantity="dry"`: virtually all non-functional — an almost deterministic signal; (2) `payment="never pay"`: >50% non-functional, confirming that without funding there is no maintenance; (3) `water_quality="fluoride abandoned"/"unknown"`: >60% non-functional, reflecting zones abandoned due to water quality; (4) `management_group="other"`: undefined management associated with poorer maintenance. These patterns justify creating the `qty_pay_combo` interaction variable during feature engineering.


The stacked bar charts confirm the discriminative power of categorical variables:

- **quantity = "dry"**: close to 100% non-functional — the strongest signal in the dataset, justifying `quantity` appearing among the top-3 variables in model importance.
- **payment = "never pay"**: non-functional rate exceeds 50%, consistent with the hypothesis that without a payment system there are no resources for maintenance.
- **management_group = "other"**: high proportion of failures, indicating undefined or informal management is associated with poorer maintenance.
- **water_quality = "fluoride abandoned" / "salty abandoned"**: very high failure rates, which makes physical sense (poor water quality degrades pump mechanisms).

As a next step, these variables will be retained and the `quantity × payment` interaction will be created as a new feature (`qty_pay_combo`).


### 2.10 Automated EDA with Sweetviz

Sweetviz generates an interactive HTML report with a single command, covering univariate distributions, variable correlations and class-based comparisons. It is compatible with Python 3.14 and requires no additional dependencies beyond the package itself.

The resulting report (`eda_report_pump_it_up.html`) can be opened in any browser and explored fully interactively without needing a server.


In [ ]:
# Sweetviz: installation and import in the same cell.
#
# Sweetviz is the Python 3.14-compatible alternative to ydata-profiling.
# Generates an interactive HTML report with distributions, correlations and
# class-based comparisons in a single call.
#
# It is installed here (not in the initial installation cell) to ensure
# the import happens in the same execution context, avoiding the
# ModuleNotFoundError that appears when pip installs in a prior step.

import subprocess, sys, importlib

try:
    importlib.import_module('sweetviz')
    print('sweetviz was already installed.')
except ImportError:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'sweetviz', '--quiet'],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-300:])
    print('sweetviz installed.')

importlib.invalidate_caches()
sv = importlib.import_module('sweetviz')
print(f'sweetviz {sv.__version__}')

print('Generating general report...')
sv.analyze(df, pairwise_analysis='off').show_html(
    'eda_report_pump_it_up.html', open_browser=False)
print('  -> eda_report_pump_it_up.html')

print('Generating comparative report functional vs non-functional...')
sv.compare(
    [df[df.status_group=='functional'].drop(columns=['status_group']),     'Functional'],
    [df[df.status_group=='non functional'].drop(columns=['status_group']), 'Non Functional'],
    pairwise_analysis='off'
).show_html('eda_report_comparativo.html', open_browser=False)
print('  -> eda_report_comparativo.html')
print('Done.')


In [ ]:
# Variables with thousands of distinct values (funder, installer, subvillage, ward)
# present the well-known 'high cardinality' problem: if one-hot encoded,
# hundreds of sparse binary columns are generated, increasing the problem
# dimensionality without adding useful information.
# The standard strategy is to group infrequent values into an
# 'other' category and only work with the most represented ones.

for col in ['funder', 'installer', 'wpt_name', 'subvillage', 'ward', 'scheme_name', 'lga']:
    n_total  = df[col].nunique()
    top10    = df[col].value_counts().head(10)
    pct_top10 = top10.sum() / len(df) * 100
    print(f'{col:<20} {n_total:>5} unique values | top-10 covers {pct_top10:.1f}% of rows')


Variables such as `wpt_name` (installation name) or `subvillage` have thousands of distinct values and the top-10 covers only a small fraction of the total. These columns contribute little to the model and will be dropped or aggressively simplified. In contrast, `funder` and `installer` have less dispersed distributions and it may be useful to keep the 50 or 100 most frequent values.


---

## 3. Feature engineering and transformations

With the exploratory analysis and baseline model results, informed decisions can be made about how to transform the data to improve performance.


### 3.1 Transformation decisions

The decisions adopted are as follows, with their justification:

**Columns dropped:**
- `id`, `recorded_by`: identifier / zero-variance
- `wpt_name`, `subvillage`, `ward`, `scheme_name`: extreme cardinality (thousands of values). Net gain from target-encoding these is marginal versus the leakage risk in CV, so they are dropped *(improvement f — justified by importance analysis in §5.2)*
- Redundant: `payment_type`, `quantity_group`, `source_type`, `extraction_type`, `waterpoint_type_group`

**Zeros treated as NaN:** `longitude`, `latitude`, `construction_year`, `population`, `gps_height`, `amount_tsh` — physically impossible or data-entry sentinel.

**Imputation (improvement b):** median computed *exclusively on the training split* and stored in `TRAIN_MEDIANS`. Test values are filled with the same stored medians so no test-set information leaks.

**New variable — `pump_age`:** `year_recorded − construction_year`.

**Logarithmic transformation (improvement a — now implemented):** `log_population = log1p(population)` and `log_amount_tsh = log1p(amount_tsh)`.  Both columns have severe right-skew (70–83 % zeros); the transform compresses the tail and makes the non-zero range more readable for trees.

**Cardinality reduction + encoding (improvements a & c):**
- `funder`, `installer`, `lga`: keep Top-50 values *per training set*, remap others → `'other'`, then LabelEncode. This eliminates cardinaliy-bias while retaining signal from major actors.
- `region_fail_rate`: mean-target encoding of `region` — fraction of non-functional pumps, computed on train only (stored in `REGION_FAIL_MAP`), then applied to test by lookup. No leakage.
- `qty_pay_combo`: interaction string `quantity + '__' + payment`, then LabelEncoded. Captures the "dry + never pay ≈ non-functional" combination.
- All other categoricals: LabelEncoder vocabulary built from the train+test union string set (safe — no imputation stats involved), same as before.


In [ ]:
# ── 3.2  Fit imputers and encoders on train only ────────────────────────────
import pandas as pd
#
# All imputation medians, cardinality vocabularies, and target-encoding maps
# are derived EXCLUSIVELY from training-set rows (improvement b).
# The test set only receives these fitted values — it never influences them.
#
# `combined` is still built for the LabelEncoder vocabulary step (we need to
# guarantee that a string seen in test gets the same integer as in train;
# that requires knowing both sets' string universes — but NO numeric stats).

combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)
print(f'Combined vocab reference (train + test): {combined.shape}')

# ── (b) Train-only imputation medians ────────────────────────────────────────
_tr_raw = train_features.copy()
_tr_raw['date_recorded'] = pd.to_datetime(_tr_raw['date_recorded'])
_tr_raw['year_recorded']  = _tr_raw['date_recorded'].dt.year
for col in ['longitude','latitude','construction_year','population','gps_height','amount_tsh']:
    _tr_raw[col] = _tr_raw[col].replace(0, np.nan)

TRAIN_MEDIANS = {c: _tr_raw[c].dropna().median()
                 for c in _tr_raw.select_dtypes(include='number').columns}
print(f'Imputation medians fit on {len(_tr_raw):,} training rows.')
del _tr_raw

# ── (a/c) Top-50 vocabulary for funder / installer / lga — train only ────────
TOP50 = {}
for col in ['funder', 'installer', 'lga']:
    TOP50[col] = set(train_features[col].value_counts().head(50).index.tolist())
print('Top-50 vocabularies built for:', list(TOP50.keys()))

# ── (a) Region target-encoding map — train only ───────────────────────────────
_tr_labels = df[['status_group']].copy()
_tr_labels['region'] = train_features['region'].values
REGION_FAIL_MAP = (
    _tr_labels.groupby('region')['status_group']
    .apply(lambda x: (x == 'non functional').mean())
    .to_dict()
)
_global_fail = (df['status_group'] == 'non functional').mean()
print(f'region_fail_rate: {len(REGION_FAIL_MAP)} regions, '
      f'global fallback = {_global_fail:.3f}')
del _tr_labels


# ── (c) Combined string vocabulary per categorical column ────────────────────
# Only the string UNIVERSE (train + test) is stored — no numeric statistics —
# so a category present only in test still maps to a stable integer.
COMBINED_VOCAB = {}
for c in combined.select_dtypes(include=['object', 'bool']).columns:
    COMBINED_VOCAB[c] = sorted(set(
        combined[c].fillna('missing').astype(str).tolist() + ['missing', 'other']
    ))
print(f'Combined vocabularies stored for {len(COMBINED_VOCAB)} categorical columns.')


### 3.3 `feature_pipeline.py` — single source of truth

The full `engineer_features` transformation is written to a standalone Python module instead of living only inside a notebook cell. Three notebooks need the exact same transformation (this one to build the matrices, notebook 03 to transform a single pump entered in the Gradio predictor), and duplicating the code would eventually cause silent divergence.

The module reads its fitted parameters (medians, vocabularies, encoding maps) from `artifacts/feature_artifacts.pkl`, so importing it anywhere reproduces the pipeline deterministically.

In [ ]:
# ── 3.3  Write feature_pipeline.py and the fitted artifacts ──────────────────
import pickle
from pathlib import Path

ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

artifacts = {
    'TRAIN_MEDIANS':   TRAIN_MEDIANS,
    'TOP50':           TOP50,
    'REGION_FAIL_MAP': REGION_FAIL_MAP,
    'global_fail':     _global_fail,
    'COMBINED_VOCAB':  COMBINED_VOCAB,
}
with open(ARTIFACTS_DIR / 'feature_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print(f'Artifacts saved: {ARTIFACTS_DIR / "feature_artifacts.pkl"}')

PIPELINE_CODE = '''\
"""feature_pipeline.py — generated by 01_data_preparation.ipynb.

Single source of truth for the feature-engineering transformation.
All fitted statistics live in artifacts/feature_artifacts.pkl; this module
only contains the transformation logic. Import from any notebook:

    from feature_pipeline import load_artifacts, engineer_features
    load_artifacts()
    X = engineer_features(raw_df)
"""
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

_ART = None

def load_artifacts(path=None):
    global _ART
    if path is None:
        here = Path(__file__).resolve().parent
        path = here / 'artifacts' / 'feature_artifacts.pkl'
    with open(path, 'rb') as f:
        _ART = pickle.load(f)
    return _ART

def engineer_features(df_input, artifacts=None):
    art = artifacts if artifacts is not None else (_ART or load_artifacts())
    TRAIN_MEDIANS   = art['TRAIN_MEDIANS']
    TOP50           = art['TOP50']
    REGION_FAIL_MAP = art['REGION_FAIL_MAP']
    _global_fail    = art['global_fail']
    COMBINED_VOCAB  = art['COMBINED_VOCAB']

    df_out = df_input.copy()

    # Date
    df_out['date_recorded'] = pd.to_datetime(df_out['date_recorded'])
    df_out['year_recorded']  = df_out['date_recorded'].dt.year
    df_out['month_recorded'] = df_out['date_recorded'].dt.month
    df_out.drop(columns=['date_recorded'], inplace=True)

    # Drop columns
    drop_cols = ['id', 'recorded_by',
                 'wpt_name', 'subvillage', 'ward', 'scheme_name',
                 'payment_type', 'quantity_group', 'source_type',
                 'extraction_type', 'waterpoint_type_group']
    df_out.drop(columns=[c for c in drop_cols if c in df_out.columns], inplace=True)

    # Impossible zeros -> NaN
    for col in ['longitude','latitude','construction_year','population','gps_height','amount_tsh']:
        if col in df_out.columns:
            df_out[col] = df_out[col].replace(0, np.nan)

    # Impute numerics with train-only medians
    for c in df_out.select_dtypes(include='number').columns:
        if df_out[c].isnull().any():
            df_out[c] = df_out[c].fillna(TRAIN_MEDIANS.get(c, df_out[c].median()))

    # Derived features
    df_out['pump_age'] = (df_out['year_recorded'] - df_out['construction_year']).clip(lower=0)
    df_out['log_population'] = np.log1p(df_out['population'])
    df_out['log_amount_tsh'] = np.log1p(df_out['amount_tsh'])

    if 'region' in df_out.columns:
        df_out['region_fail_rate'] = df_out['region'].map(REGION_FAIL_MAP).fillna(_global_fail)

    if 'quantity' in df_out.columns and 'payment' in df_out.columns:
        df_out['qty_pay_combo'] = (df_out['quantity'].fillna('missing').astype(str)
                                    + '__' + df_out['payment'].fillna('missing').astype(str))

    # Top-50 cardinality reduction
    for col in ['funder', 'installer', 'lga']:
        if col in df_out.columns:
            vocab = TOP50.get(col, set())
            df_out[col] = (df_out[col].fillna('missing')
                           .apply(lambda x: x if x in vocab else 'other'))

    # LabelEncode remaining categoricals with the stored combined vocabulary
    for c in df_out.select_dtypes(include=['object', 'bool']).columns:
        df_out[c] = df_out[c].fillna('missing').astype(str)
        vocab = COMBINED_VOCAB.get(c)
        le = LabelEncoder()
        if vocab is not None:
            le.fit(vocab)
        else:
            le.fit(sorted(set(df_out[c].tolist() + ['missing', 'other'])))
        df_out[c] = df_out[c].apply(lambda x: x if x in le.classes_ else 'other')
        df_out[c] = le.transform(df_out[c])

    return df_out
'''

with open('feature_pipeline.py', 'w', encoding='utf-8') as f:
    f.write(PIPELINE_CODE)
print('Module written: feature_pipeline.py')


In [ ]:
# ── 3.3b  Apply the pipeline to train and test ────────────────────────────────
import importlib
import feature_pipeline
importlib.reload(feature_pipeline)                # pick up freshly written file
from feature_pipeline import load_artifacts, engineer_features

load_artifacts(ARTIFACTS_DIR / 'feature_artifacts.pkl')

y          = df['status_group']
X_eng      = engineer_features(df.drop(columns=['status_group']))
X_test_eng = engineer_features(test_features)

print(f'Features after engineering: {X_eng.shape[1]} columns')
new_feats = [c for c in X_eng.columns
             if c in ('pump_age','log_population','log_amount_tsh',
                      'month_recorded','qty_pay_combo','region_fail_rate')]
print(f'New/transformed features added: {new_feats}')
print('All columns:', list(X_eng.columns))


### 3.4 Transformation verification

In [ ]:
# Verify that no nulls or non-numeric types remain after transformation.
assert X_eng.isnull().sum().sum() == 0,       'Nulls remain in train'
assert X_test_eng.isnull().sum().sum() == 0,  'Nulls remain in test'
assert list(X_eng.columns) == list(X_test_eng.columns), 'Column mismatch train/test'
assert len(X_eng) == len(y), 'Row count mismatch vs labels'

# Check new features exist
for feat in ['pump_age', 'log_population', 'log_amount_tsh', 'region_fail_rate', 'qty_pay_combo']:
    assert feat in X_eng.columns, f'Expected feature missing: {feat}'

# Log features must be non-negative and finite
for col in ['log_population', 'log_amount_tsh']:
    assert (X_eng[col] >= 0).all() and np.isfinite(X_eng[col]).all(), f'{col} invalid'

# High-card columns dropped
for col in ['wpt_name', 'subvillage', 'ward', 'scheme_name']:
    assert col not in X_eng.columns, f'Column should have been dropped: {col}'

print('All assertions passed.')
print(f'Train: {X_eng.shape}  |  Test: {X_test_eng.shape}')
print('New features present:', [c for c in X_eng.columns
      if c in ('pump_age','log_population','log_amount_tsh',
               'month_recorded','qty_pay_combo','region_fail_rate')])


In [ ]:
# The distribution of the new pump_age variable is visualised to validate
# that it makes sense and effectively discriminates between classes.

df_vis = X_eng[['pump_age']].copy()
df_vis['status_group'] = y.values
df_vis = df_vis[df_vis['pump_age'].between(0, 60)]

fig, ax = plt.subplots(figsize=(9, 4))
for label, color in palette.items():
    subset = df_vis[df_vis['status_group'] == label]['pump_age']
    subset.plot.kde(ax=ax, label=label, color=color, linewidth=1.8)

ax.set_title('Pump age distribution by status (derived variable)', fontsize=11)
ax.set_xlabel('Years since installation')
ax.legend(fontsize=9)
for _ax in (np.array([ax]).flatten() if hasattr(ax,'__len__') else [ax]):
    _ax.grid(False)
    for _sp in _ax.spines.values(): _sp.set_visible(False)
plt.tight_layout()
plt.savefig('fig_pump_age_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


The distribution confirms the expected pattern: non-functional pumps tend to be older, while functional ones concentrate at more recent ages. The `pump_age` variable captures this relationship directly.


---

## 4. Persist artifacts for notebooks 02 and 03

Everything the downstream notebooks need is written to `artifacts/`:

| File | Consumed by | Content |
|---|---|---|
| `X_eng.parquet` | 02, 03 | Engineered training feature matrix |
| `X_test_eng.parquet` | 02 | Engineered competition-test matrix |
| `y.parquet` | 02 | Target labels (`status_group`) |
| `feature_artifacts.pkl` | 03 (via `feature_pipeline.py`) | Fitted medians, vocabularies, encoding maps |

`02_model_training.ipynb` starts by reading these files — no EDA or feature engineering is re-executed there.

In [ ]:
# ── 4  Save the engineered matrices ───────────────────────────────────────────
X_eng.to_parquet(ARTIFACTS_DIR / 'X_eng.parquet')
X_test_eng.to_parquet(ARTIFACTS_DIR / 'X_test_eng.parquet')
y.to_frame('status_group').to_parquet(ARTIFACTS_DIR / 'y.parquet')

for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f'  {f.name:<28} {f.stat().st_size/1024:,.0f} KB')
print('\nPart 1 complete — continue in 02_model_training.ipynb')
